<a href="https://colab.research.google.com/github/nermal1/Stock-Market-Prediction-437/blob/main/RandomForestRealPred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

## Setup

In [ ]:
tickers = ['AAPL', 'MSFT', '^GSPC', '^DJI']
results = {}

## Indicator Functions

In [ ]:
def weighted_moving_average(data, period):
  weights = np.arange(1, period + 1)
  return data.rolling(period).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)

In [ ]:
def featureSelection(df):

  df = df.copy()

  # Basic Returns
  df['Return'] = df['Close'].pct_change()

  # Technical Indicators
  df['SMA_14'] = df['Close'].rolling(window=14).mean()
  df['SMA_50'] = df['Close'].rolling(window=50).mean()
  df['WMA_14'] = weighted_moving_average(df['Close'], 14)
  df['Momentum_10'] = df['Close'] / df['Close'].shift(10) - 1
  df['Volatility_14'] = df['Return'].rolling(window=14).std()

  # RSI Calculation
  delta = df['Close'].diff()
  gain = (delta.where(delta > 0, 0))
  loss = (-delta.where(delta < 0, 0))
  avg_gain = gain.rolling(window=14).mean()
  avg_loss = loss.rolling(window=14).mean()
  rs = avg_gain / avg_loss
  df['RSI_14'] = 100 - (100 / (1 + rs))

  # Lags (Previous days' returns)
  lags = [1, 2, 3, 5]
  for lag in lags:
    df[f'Lag_{lag}'] = df['Return'].shift(lag)

  # Target: 1 if Up, 0 if Down
  df['Target'] = np.where(df['Return'] > 0, 1, 0)

  return df.dropna()

## Data pipeline

In [ ]:
for ticker in tickers:
  print(f"Fetching row for {ticker}")
  raw_df = yf.download(ticker, start="2010-01-01", end="2019-12-31", progress=False, auto_adjust=True)

  if isinstance(raw_df.columns, pd.MultiIndex):
    raw_df = raw_df.xs(ticker, axis=1, level=1)

  results[ticker] = featureSelection(raw_df)



Fetching row for AAPL
Fetching row for MSFT
Fetching row for ^GSPC
Fetching row for ^DJI


In [ ]:
print("\nStarting Random Forest Optimization")

feature_pool = ['RSI_14', 'SMA_14', 'SMA_50', 'WMA_14', 'Momentum_10', 'Volatility_14', 'Lag_1', 'Lag_2', 'Lag_5']

n_estimators_list = [50, 100, 150]

for ticker, df in results.items():
  print(f"\nOptimizing for {ticker}")
  y = df['Target']

  best_accuracy = 0
  best_combo = []
  best_n_trees = 50
  best_metrics = {}
  model_count = 0

  for r in range(1, 5):
    print(f"  Testing feature sets of size {r}")

    for combo in combinations(feature_pool, r):
      combo_list = list(combo)
      X = df[combo_list]

      # Split Data (Time Series Split - No Shuffle)
      X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

      # Loop through n_estimators
      for n_trees in n_estimators_list:
        model = RandomForestClassifier(n_estimators=n_trees, random_state=42, n_jobs=-1)

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)

        # Print every 50 models trained
        model_count += 1
        if model_count % 50 == 0:
          print(f"> Trained {model_count} models (Current Best: {best_accuracy:.2%})")

          if acc > best_accuracy:
            best_accuracy = acc
            best_combo = combo_list
            best_n_trees = n_trees
            best_metrics = {
              'Precision': precision_score(y_test, y_pred, zero_division=0),
              'Recall': recall_score(y_test, y_pred, zero_division=0),
              'F1': f1_score(y_test, y_pred, zero_division=0)
              }

  print(f"Results for {ticker}:")
  print(f"  Best Accuracy:  {best_accuracy:.2%}")
  print(f"  Best n_trees:   {best_n_trees}")
  print(f"  Best Features:  {best_combo}")
  print(f"  Precision:      {best_metrics['Precision']:.4f}")
  print(f"  Recall:         {best_metrics['Recall']:.4f}")
  print(f"  F1 Score:       {best_metrics['F1']:.4f}")


Starting Random Forest Optimization

Optimizing for AAPL
  Testing feature sets of size 1
  Testing feature sets of size 2
> Trained 50 models (Current Best: 0.00%)
> Trained 100 models (Current Best: 54.25%)
  Testing feature sets of size 3
> Trained 150 models (Current Best: 54.25%)
> Trained 200 models (Current Best: 54.25%)
> Trained 250 models (Current Best: 59.51%)
> Trained 300 models (Current Best: 59.51%)
> Trained 350 models (Current Best: 59.51%)
  Testing feature sets of size 4
> Trained 400 models (Current Best: 59.51%)
> Trained 450 models (Current Best: 59.51%)
> Trained 500 models (Current Best: 59.51%)
> Trained 550 models (Current Best: 59.51%)
> Trained 600 models (Current Best: 59.51%)
> Trained 650 models (Current Best: 59.51%)
> Trained 700 models (Current Best: 59.51%)
> Trained 750 models (Current Best: 59.51%)
Results for AAPL:
  Best Accuracy:  59.51%
  Best n_trees:   100
  Best Features:  ['RSI_14', 'Momentum_10', 'Lag_5']
  Precision:      0.6212
  Recall: